# Aula 04 — Persistência e Streaming com LangGraph

Este notebook demonstra dois conceitos fundamentais do LangGraph:

- **Persistência de estado (Checkpointing):** o grafo salva um snapshot do `AgentState` após cada passo, usando SQLite como backend. Isso permite retomar conversas interrompidas e manter histórico entre chamadas separadas.
- **Streaming de eventos:** em vez de aguardar a resposta completa do modelo, iteramos sobre os eventos do grafo em tempo real com `graph.stream()`, recebendo cada nó (`llm`, `action`) conforme ele termina.
- **Memória de thread:** o conceito de `thread_id` dentro do `configurable` isola contextos de conversa distintos, permitindo múltiplas sessões paralelas independentes.


In [ ]:
# Instalação das dependências necessárias:
# - langgraph: framework de grafos de agentes
# - langchain-google-genai: integração com o Gemini da Google
# - langchain-tavily: ferramenta de busca na web
# - aiosqlite: driver assíncrono do SQLite para o checkpointer do LangGraph
%pip install python-dotenv
%pip install -U langgraph langchain langchain-google-genai langchain-tavily python-dotenv aiosqlite

## 1. Configuração de Variáveis de Ambiente

As chaves de API são carregadas de um arquivo `.env` local (nunca hardcoded no código).
- `GEMINI_API_KEY` → mapeada para `GOOGLE_API_KEY`, que é o nome esperado pelo `langchain-google-genai`
- `TAVILY_API_KEY` → usada diretamente pelo `TavilySearch`


In [ ]:
import os
from dotenv import load_dotenv

# Carrega as variáveis do arquivo .env para os.environ
load_dotenv()

# langchain-google-genai espera a chave em GOOGLE_API_KEY;
# fazemos o mapeamento aqui para manter o nome GEMINI_API_KEY no .env
os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI_API_KEY') 
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

## 2. Imports

Importamos os blocos principais do LangGraph e LangChain:

| Import | Papel |
|--------|-------|
| `operator` | Fornece `operator.add` como *reducer* do estado |
| `Annotated` | Permite anexar metadados de comportamento a tipos (PEP 593) |
| `StateGraph, END` | Núcleo do LangGraph: grafo dirigido com estado tipado |
| `*Message` | Tipos de mensagem do LangChain (HumanMessage, SystemMessage, etc.) |
| `SqliteSaver` | Checkpointer que persiste snapshots do estado em um banco SQLite |
| `TypedDict` | Base para definir o schema do estado do grafo como dicionário tipado |


In [ ]:
import operator
from typing import Annotated, List, Any, Dict
from dataclasses import dataclass, field
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, BaseMessage, AnyMessage

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

# SqliteSaver: implementa a interface de checkpointing do LangGraph
# usando SQLite como storage. Alternativas: MemorySaver (volátil),
# PostgresSaver (produção), RedisSaver (cache)
from langgraph.checkpoint.sqlite import SqliteSaver

from typing_extensions import TypedDict

import sqlite3

## 3. Estado do Agente e Configuração de Persistência

### AgentState

O `AgentState` é o schema de estado compartilhado entre todos os nós do grafo.

```python
messages: Annotated[list[AnyMessage], operator.add]
```

A anotação `Annotated[..., operator.add]` instrui o LangGraph a usar `operator.add` como **reducer**: quando um nó retorna `{'messages': [nova_msg]}`, o framework *concatena* essa lista ao estado existente em vez de substituí-lo. Sem o reducer, cada nó sobrescreveria todo o histórico de mensagens.

### SqliteSaver

O `check_same_thread=False` é necessário porque o SQLite por padrão bloqueia acesso de threads distintas. Como o LangGraph pode executar nós em threads diferentes, essa restrição precisa ser relaxada.


In [ ]:
class AgentState(TypedDict):
    # Lista de mensagens com reducer 'operator.add':
    # cada nó ADICIONA mensagens ao histórico (nunca substitui)
    messages: Annotated[list[AnyMessage], operator.add]

# Conexão SQLite com check_same_thread=False para permitir
# acesso concorrente de múltiplas threads do LangGraph
conn = sqlite3.connect("checkpoints.db", check_same_thread=False)

# memory é o checkpointer: salva um snapshot do AgentState
# após cada passo do grafo, identificado por thread_id
memory = SqliteSaver(conn)

## 4. Classe Agent — Construção do Grafo ReAct

A classe `Agent` encapsula a lógica de construção e execução do grafo. O padrão implementado é o **ReAct** (Reasoning + Acting):

```
  [entrada] ──► [nó: llm] ──► tem tool_call? ──► SIM ──► [nó: action] ──┐
                    ▲                                                      │
                    └──────────────────────────────────────────────────────┘
                                   │ NÃO
                                   ▼
                                  END
```

O grafo é compilado com `checkpointer=checkpointer`, o que habilita a persistência automática de estado a cada transição de nó.


In [ ]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        # Armazena o prompt de sistema para injetar em cada chamada ao LLM
        self.system = system
        
        # Inicializa o grafo tipado com o schema AgentState
        graph = StateGraph(AgentState)

        # Nó 'llm': chama o modelo de linguagem com o histórico atual
        graph.add_node("llm", self.call_gemini)

        # Nó 'action': executa as tool_calls solicitadas pelo LLM
        graph.add_node("action", self.take_action)

        # Aresta condicional: se o LLM solicitou ferramentas → vai para 'action'
        # caso contrário → encerra o grafo (END)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})

        # Após executar as ferramentas, sempre retorna ao nó 'llm'
        # (o LLM processa os resultados e decide se precisa de mais ações)
        graph.add_edge("action", "llm")

        # Define 'llm' como ponto de entrada do grafo
        graph.set_entry_point("llm")

        # Compila o grafo com o checkpointer: habilita persistência automática
        # de snapshots do AgentState após cada transição de nó
        self.graph = graph.compile(checkpointer=checkpointer)
        
        # Dicionário nome→ferramenta para lookup rápido em take_action
        self.tools = {t.name: t for t in tools}
        
        # Vincula as ferramentas ao modelo: o LLM receberá os schemas
        # das ferramentas e poderá gerar tool_calls estruturadas
        self.model = model.bind_tools(tools)

    def call_gemini(self, state: AgentState):
        """Nó 'llm': invoca o Gemini com o histórico completo de mensagens."""
        messages = state['messages']
        
        # Se há um system prompt, insere-o no início do histórico a cada chamada.
        # Nota: o system prompt NÃO é persistido no AgentState —
        # ele é injetado dinamicamente em runtime para não poluir o histórico.
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages

        print("Mensagens enviadas ao modelo:", messages)
        message = self.model.invoke(messages)
        
        # Retorna a resposta para ser ADICIONADA ao estado via reducer operator.add
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        """Função de roteamento condicional: verifica se a última mensagem
        contém tool_calls (pedidos de execução de ferramentas)."""
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        """Nó 'action': executa todas as tool_calls da última mensagem do LLM
        e empacota os resultados como ToolMessages para retornar ao grafo."""
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling Tool: {t['name']} with args: {t['args']}")
            
            # Lookup no dicionário de ferramentas e invocação com os args do LLM
            result = self.tools[t['name']].invoke(t['args'])
            
            # ToolMessage vincula o resultado à tool_call pelo 'id',
            # permitindo que o LLM rastreie qual resultado pertence a qual chamada
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Returning to LLM after action!")
        return {'messages': results}

## 5. Instanciação do Agente

Configuramos e instanciamos o agente com:
- **TavilySearch** como ferramenta de busca na web (máx. 3 resultados)
- **Gemini 1.5 Pro** como modelo de linguagem
- **`memory`** (SqliteSaver) como checkpointer
- Um **system prompt** em português com instrução explícita para usar o histórico da conversa em comparações


In [ ]:
current_tavily_api_key = os.getenv('TAVILY_API_KEY')
if not current_tavily_api_key:
    raise ValueError("TAVILY_API_KEY não encontrada. Certifique-se de que está no seu .env e python-dotenv está instalado.")

# Ferramenta de busca web: retorna até 3 resultados por query
tool = TavilySearch(max_results=3, tavily_api_key=current_tavily_api_key)


# System prompt: define o comportamento e as regras de uso das ferramentas.
# A última linha instrui o agente a usar o HISTÓRICO da conversa (checkpointed)
# para responder comparações sem precisar buscar novamente.
prompt_system = """Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.
Você tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).
Busque informações apenas quando tiver certeza do que procurar.
Se precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.
Quando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas."""

# Instancia o modelo Gemini 1.5 Pro via LangChain
model = ChatGoogleGenerativeAI(model="gemini-1.5-pro") 

# Cria o agente completo: grafo ReAct + Tavily + persistência SQLite
abot = Agent(model, [tool], system=prompt_system, checkpointer=memory)

## 6. Demonstração de Persistência — Thread 1

### O que é `thread_id`?

O `thread_id` é o identificador que o checkpointer usa para separar contextos de conversa. Toda chamada com `thread_id: "1"` compartilha o mesmo histórico persistido no SQLite.

```python
thread = {"configurable": {"thread_id": "1"}}
```

### Como `graph.stream()` funciona

Em vez de `graph.invoke()` (bloqueante), `graph.stream()` é um gerador que emite um evento a cada transição de nó. Cada evento é um dicionário `{nome_do_nó: estado_resultante}`, permitindo ver o raciocínio do agente em tempo real.

As três células seguintes demonstram que perguntas subsequentes na mesma thread **herdam o contexto anterior** — o agente sabe que SP tem 31°C porque a busca foi feita numa chamada anterior e persistida.


In [ ]:
messages = [HumanMessage(content="Como está o tempo em São Paulo hoje (21/08/2025)?")]
# thread_id '1': todas as mensagens desta sessão serão persistidas juntas no SQLite
thread = {"configurable": {"thread_id": "1"}} 


print("\n--- Pergunta 1: Tempo em São Paulo ---")
# graph.stream() emite eventos por nó à medida que o grafo executa
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        # Filtramos apenas 'llm' (resposta do modelo) e 'action' (resultado das ferramentas)
        if k in ("llm", "action"): 
             print(f"{k}: {v['messages']}")

In [ ]:
# Mesma thread_id '1': o checkpointer carrega o histórico do SQLite
# antes da execução, incluindo a resposta sobre SP da célula anterior.
# O agente pode responder 'E no Rio?' sabendo que o contexto é sobre tempo.
messages = [HumanMessage(content="E no Rio de Janeiro?")]
thread = {"configurable": {"thread_id": "1"}} 

print("\n--- Pergunta 2: Tempo no Rio de Janeiro ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")

In [ ]:
# Pergunta ambígua que só faz sentido com memória persistida:
# o agente usa SP (31°C) e RJ (27°C) do histórico da thread '1'
# para responder sem precisar buscar novamente.
messages = [HumanMessage(content="Qual está mais quente?")]
thread = {"configurable": {"thread_id": "1"}} 

print("\n--- Pergunta 3: Comparação ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")

## 7. Demonstração de Isolamento — Thread 2

Esta célula usa `thread_id: "2"` — uma sessão completamente nova e isolada.

O agente **não tem acesso** ao histórico da thread `"1"`, portanto a pergunta `"Qual está mais quente?"` não tem contexto e o modelo solicita esclarecimentos. Isso prova que o `thread_id` funciona como chave de isolamento: cada thread tem seu próprio histórico independente no banco de dados.


In [ ]:
messages = [HumanMessage(content="Qual está mais quente?")]
# thread_id '2': contexto DIFERENTE e ISOLADO da thread '1'.
# O checkpointer não encontra histórico aqui — o agente não tem
# como saber 'mais quente do que quê?' e pede esclarecimento.
thread = {"configurable": {"thread_id": "2"}} 

print("\n--- Pergunta 4: Comparação ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")